### Notebook for ETL of indicators for individual works  

-  number of references  
-  number of pages  
-  references/page  
-  fwci from oa  
-  fwci from corpus  
-  hc status from oa  
-  hc status from corpus  
-  copied references  
-  copied reference ratio  
-  disruption index  
-  citer authors FD  
-  cited authors FD  
-  citer institutions FD  
-  cited institutions FD  
-  self references  
-  self referencing rate per reference  
-  citer topics FD  
-  cited topics FD  
-  journal spectral rank  
-  institution spectral rank  


In [111]:
%run common_setup.ipynb

In [112]:
from dataclasses import dataclass
@dataclass
class WorkIndicators:

    work_id: str = ''
    number_of_references: int = 0
    number_of_pages: int = 0
    
    def references_per_page(self) -> float:
        return self.number_of_references/self.number_of_pages


#### This cell extracts Works data and transforms it to the work_indicators

- Construct time-series of citations from reference lists  
- Compute the centile for each publication year
- Filter highly cited papers  
- Group the authors of hte highly cited papers

In [113]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def references_per_page(self):
        sql = """
            SELECT id AS work_id,
                    referenced_works_count,
                    try_cast("biblio.last_page" AS INT) - try_cast("biblio.first_page" AS INT) AS page_count,
                    referenced_works_count/page_count AS references_per_page
                FROM project.raw
            """
        self.db.sql(sql).show()
        return
    
    def fwci(self):
        sql = """
            WITH
            get_fwci_endogenous_CTE AS
                (SELECT DISTINCT cited_id,
                                count(citer_id) OVER (PARTITION BY cited_id)/
                                    (count(citer_id) OVER (PARTITION BY cited_year)/
                                    count(DISTINCT citer_id) OVER (PARTITION BY cited_year)) AS fwci_endogenous,
                FROM project.citer_cited
                ORDER BY fwci_endogenous DESC
                )
            SELECT id AS cited_id,
                    fwci,
                    fwci_endogenous
                FROM project.raw
                LEFT JOIN get_fwci_endogenous_CTE
                ON id = cited_id
            ORDER BY fwci DESC
            """
        self.db.sql(sql).show()
        return
    
    def highly_cited(self):
        sql = """  
            -- ETL FOR highly_cited_work
            -- =========================
            WITH
            citation_counts_CTE AS
                (SELECT DISTINCT cited_by_count,
                        count(citer_id) OVER (PARTITION BY cited_id) AS cited_by_count_endogenous,
                        cited_id,
                        cited_year,
                FROM project.citer_cited
                LEFT JOIN project.raw
                ON id = cited_id
                ORDER BY cited_by_count_endogenous DESC
                )

            SELECT cited_id,
                    percent_rank(ORDER BY cited_by_count) OVER w AS percent_rank_total,
                    percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_total_endogenous,
            FROM citation_counts_CTE
            WINDOW w AS (PARTITION BY cited_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
            ORDER BY percent_rank_total DESC, percent_rank_total_endogenous DESC
            """
        self.db.sql(sql).show()
        return
    
    def copied_references(self):
        sql = """
            -- ETL FOR copied_references
            -- =========================
            WITH
            get_copied_CTE AS
                (SELECT r1.citer_id,
                        count(r2.cited_id) AS copied_count,
                        referenced_works_count AS total_count,
                    FROM project.citer_cited r1
                    LEFT JOIN project.citer_cited r2
                    ON r1.cited_id = r2.citer_id
                    LEFT JOIN project.raw
                    ON id = r1.citer_id
                    WHERE r1.cited_id = r2.cited_id
                    GROUP BY ALL
                )

            SELECT citer_id,
                    total_count,
                    copied_count/total_count AS copied_fraction
            FROM get_copied_CTE
            ORDER BY copied_fraction DESC, total_count DESC
            """
        self.db.sql(sql).show()
        return
    
    def disruption_index(self):
        sql = """  
            -- ETL FOR disruption_index
            -- ========================
            WITH
            extract_work_CTE AS
                (SELECT id AS work_id,
                        fwci,
                        unnest(referenced_works) AS referenced_work
                FROM project.raw
                ),
            extract_disruption_index_CTE AS
                (SELECT work_id,
                        r1.fwci/avg(r2.fwci) OVER (PARTITION BY work_id) AS disruption_index
                FROM extract_work_CTE r1
                LEFT JOIN project.raw r2
                ON referenced_work = r2.id
                ORDER BY disruption_index DESC
                )

            SELECT DISTINCT *
            FROM extract_disruption_index_CTE
            WHERE disruption_index NOT NULL AND disruption_index != 'NaN' AND disruption_index != 'Infinity'
            ORDER BY disruption_index DESC
            """
        self.db.sql(sql).show()
        return
    
    def self_references(self):
        sql = """  
            -- ETL FOR self_references
            -- =======================
            WITH
            get_selfcited_CTE AS
                (SELECT DISTINCT cited_id,
                        isSelfCited
                FROM
                    (SELECT citer_id,
                            cited_id,
                            CASE WHEN list_has_any(citer_authors, cited_authors) = true IS true THEN 1 ELSE 0 END AS isSelfCited
                    FROM (SELECT citer_id, 
                                    list(citer.author_id) AS citer_authors,
                                    cited_id,
                                    list(cited.author_id) AS cited_authors
                            FROM project.citer_cited
                            INNER JOIN project.authorships citer
                            ON citer_id = citer.work_id
                            INNER JOIN project.authorships cited
                            ON cited_id = cited.work_id
                            GROUP BY citer_id, cited_id
                            )
                    )
                )

            SELECT count(), sum(isSelfCited), sum(1-isSelfCited)
            FROM get_selfcited_CTE
            GROUP BY ALL
            """
        self.db.sql(sql).show()
        return
    
    def citations_per_work(self):
        sql = """ 
        CREATE OR REPLACE TABLE memory.citations_per_work AS
            SELECT count(work_id) AS cited_by_count_endogenous,
                    sum(cited_by_count) AS cited_by_count_total,
                    author_id,
                    author_name,
                    publication_year
            FROM
                (SELECT DISTINCT id AS work_id,
                    w.cited_by_count,
                    unnest(authorships).author.id AS author_id,
                    unnest(authorships).author.display_name as author_name,
                    w.publication_year
                FROM project.raw w
                LEFT JOIN (SELECT id AS work_id,
                            unnest(referenced_works) AS cited_id
                            FROM project.raw           
                            ) c
                ON w.id = c.cited_id
                )
                GROUP BY author_id, author_name, publication_year
            ORDER BY cited_by_count_endogenous DESC
        """
        self.db.sql(sql)
        sql.db.sql("SELECT * FROM memory_citations_per_work").show()
        return

    def citations_per_work_ranked(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
                SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
                FROM memory.citations_per_work 
                WINDOW w AS (PARTITION BY publication_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
                ORDER BY publication_year DESC, percent_rank_total DESC
                """
        self.db.sql(sql)
        return

    def citation_summation(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations AS
                SELECT author_id,
                        author_name,
                        sum(cited_by_count_total) AS citations_total,
                        sum(cited_by_count_endogenous) AS citations_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE author_id NOT NULL
                GROUP BY author_id, author_name
                ORDER BY citations_endogenous DESC                    "biblio.last_page"
                """
        self.db.sql(sql)
        return

    def hca_summation(self):       

        sql = """ 
                CREATE OR REPLACE TABLE memory.hca_endogenous AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_endogenous                    "biblio.last_page"
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_endogenous >= 0.99
                GROUP BY ALL
                ORDER BY hca_endogenous DESC;

                CREATE OR REPLACE TABLE memory.hca_total AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_total,
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_total >= 0.99
                GROUP BY ALL
                ORDER BY hca_total DESC
                """
        self.db.sql(sql)
        return
    
    def citations_endogenous_all(self):
        sql = """ 
            CREATE OR REPLACE TABLE memory.citations_endogenous_all AS
                SELECT author_id,
                        author_name,
                        citations_total,
                        citations_endogenous,
                        hca_total,
                        hca_endogenous
                FROM memory.citations
                LEFT JOIN
                    (SELECT t.*,
                            e.hca_endogenous
                        FROM memory.hca_total t
                        LEFT JOIN memory.hca_endogenous e
                        USING (author_id)
                    ) sub
                USING (author_id, author_name)
                ORDER BY citations_endogenous DESC
            """
        self.db.sql(sql)
        return

    def author_works_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.author_works_counts AS
                SELECT au.author_id,
                        au.author_name,
                        a.first,
                        a.middle,
                        a.last,
                        a.fullname,
                        a.orcid,
                        a.display_name_alternatives,
                        count(work_id) AS works_count_endogenous,
                        works_count,
                        cited_by_count,
                        "2yr_mean_citedness",
                        h_index      
                    FROM econ.authorships au
                        LEFT JOIN econ.authors a
                        ON a.author_id = au.author_id
                    GROUP BY ALL
                    ORDER BY works_count_endogenous DESC
            """
        self.db.sql(sql)
        return
    
    def citation_summary(self):

        sql = """ 
            CREATE OR REPLACE TABLE econ.citation_summary AS
                SELECT DISTINCT c.author_id,
                        c.author_name,
                        a.author_name,
                        a.works_count_endogenous,
                        s.citations_total AS citations_total_,
                        c.citations_endogenous,
                        hca_total,
                        hca_endogenous,
                        a.orcid,
                        a.display_name_alternatives,
                        a.works_count AS works_count_total,
                        a.cited_by_count,
                        a."2yr_mean_citedness",                
                SetUp:

    def __init__(self):
        self._setup_db()
        return

        self.db.sql("SHOW ALL TABLES").show()
        return
                        a.h_index
                FROM memory.citations c
                    LEFT JOIN memory.citations_endogenous_all s
                    ON c.author_id = s.author_id
                        LEFT JOIN memory.author_works_counts a
                        ON c.author_id = a.author_id
            ORDER BY cited_by_count DESC, h_index DESC
            """
        self.db.sql(sql)
        return
    
    def show_all(self):
        self.db.sql("SELECT * FROM memory.citations_per_work").show()
        self.db.sql("SELECT * FROM memory.citations_per_work_ranked").show()
        self.db.sql("SELECT * FROM memory.citations").show()
        self.db.sql("SELECT * FROM memory.hca_endogenous").show() 
        self.db.sql("SELECT * FROM memory.citations_endogenous_all").show()
        self.db.sql("SELECT * FROM econ.authors").show()  
        self.db.sql("SELECT * FROM econ.citation_summary").show()
        return
    
    def load_citations(self):
        df = self.db.sql("""
                         SELECT * EXCLUDE (author_name_1, display_name_alternatives) FROM project.citation_summary ORDER BY hca_endogenous DESC, h_index DESC
                         """).df().reset_index(drop=True)
        df.to_excel('../DATA/citation_summary.xlsx', index=False)
        return


In [114]:
def main():

    cetl = CorpusETL()
    cetl.references_per_page()
    cetl.fwci()
    cetl.highly_cited()
    cetl.copied_references()
    cetl.disruption_index()
    cetl.self_references()
    # cetl.citations_per_work()
    # cetl.citations_per_work_ranked()
    # cetl.citation_summation()
    # cetl.hca_summation()
    # cetl.author_works_count()
    # cetl.citations_endogenous_all()
    # cetl.citation_summary()
    # cetl.show_all()
    # cetl.load_citations()
    cetl.db.close()
        
    return

In [115]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ backup   │ main    │ article_vectors      │ [article_vector, u…  │ [DOUBLE, VARCHAR, VARCHAR, BIGINT]    │ false     │
│ backup   │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR]  │ false     │
│ backup   │ main    │ both_edge_list       │ [citer_unit, cited…  │ [VARCHAR, VARCHAR, DOUBLE]            │ false     │
│ backup   │ main    │ institution_edge_l…  │ [citer_unit, cited…  │ [VARCHAR, VARCHAR, DOUBLE]            │ false     │
│ backup   │ main    │ pagerank_